# 16.5 - Monitoring & Observability

Status: VERIFIED

## What Are We Solving?

Models degrade silently. Data drift, concept drift, and infrastructure failures can go unnoticed for weeks. Monitoring catches problems before your users do.

## Mental Model

Monitoring is the vital signs monitor in a hospital. It watches prediction latency, error rates, data distribution shifts, and resource usage — alerting you when something goes wrong.

In [1]:
import matplotlib
matplotlib.use('Agg')
import logging
import json
import time
import random
from datetime import datetime

# Structured logging setup
logger = logging.getLogger('ml_service')
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))
logger.addHandler(handler)

# Simulate prediction logging
random.seed(42)
for i in range(5):
    latency = random.uniform(10, 80)
    status = 'success' if random.random() > 0.1 else 'error'
    logger.info(json.dumps({
        'event': 'prediction',
        'model_version': '1.0.0',
        'latency_ms': round(latency, 2),
        'status': status,
        'input_features': 5,
    }))


2026-08-29 12:03:51 | INFO | {"event": "prediction", "model_version": "1.0.0", "latency_ms": 54.76, "status": "error", "input_features": 5}


2026-08-29 12:03:51 | INFO | {"event": "prediction", "model_version": "1.0.0", "latency_ms": 29.25, "status": "success", "input_features": 5}


2026-08-29 12:03:51 | INFO | {"event": "prediction", "model_version": "1.0.0", "latency_ms": 61.55, "status": "success", "input_features": 5}


2026-08-29 12:03:51 | INFO | {"event": "prediction", "model_version": "1.0.0", "latency_ms": 72.45, "status": "error", "input_features": 5}


2026-08-29 12:03:51 | INFO | {"event": "prediction", "model_version": "1.0.0", "latency_ms": 39.53, "status": "error", "input_features": 5}


## Key Metrics to Track

| Category | Metric | Why |
|----------|--------|-----|
| **Performance** | Prediction latency (p50, p95, p99) | SLA compliance |
| **Reliability** | Error rate, uptime | Availability |
| **Data Quality** | Null rate, feature distribution shifts | Input drift |
| **Model Quality** | Prediction distribution, confidence scores | Concept drift |
| **Resources** | CPU, memory, GPU utilization | Cost & scaling |

In [2]:
import matplotlib
matplotlib.use('Agg')
import numpy as np

# Calculate monitoring metrics from simulated prediction logs
np.random.seed(42)
latencies = np.random.lognormal(mean=3.5, sigma=0.5, size=1000)
predictions = np.random.choice([0, 1], size=1000, p=[0.7, 0.3])

metrics = {
    'p50_latency_ms': float(np.percentile(latencies, 50)),
    'p95_latency_ms': float(np.percentile(latencies, 95)),
    'p99_latency_ms': float(np.percentile(latencies, 99)),
    'mean_latency_ms': float(np.mean(latencies)),
    'total_predictions': len(predictions),
    'positive_rate': float(np.mean(predictions)),
    'throughput_rps': 1000 / (np.sum(latencies) / 1000),
}

print("Monitoring Dashboard:")
print("=" * 40)
for key, value in metrics.items():
    if 'rate' in key:
        print(f"  {key:25s}: {value:.3f}")
    elif 'throughput' in key:
        print(f"  {key:25s}: {value:.1f} req/s")
    else:
        print(f"  {key:25s}: {value:.2f} ms")


Monitoring Dashboard:
  p50_latency_ms           : 33.54 ms
  p95_latency_ms           : 76.59 ms
  p99_latency_ms           : 105.43 ms
  mean_latency_ms          : 37.78 ms
  total_predictions        : 1000.00 ms
  positive_rate            : 0.301
  throughput_rps           : 26.5 req/s


## Data Drift Detection

Compare the distribution of incoming features against the training distribution. A common approach is the Population Stability Index (PSI).

In [3]:
import matplotlib
matplotlib.use('Agg')
import numpy as np

# Simple PSI calculation
def calculate_psi(reference, current, bins=10):
    """Population Stability Index between two distributions."""
    eps = 1e-4
    breakpoints = np.linspace(
        min(reference.min(), current.min()),
        max(reference.max(), current.max()),
        bins + 1
    )
    ref_pct = np.histogram(reference, bins=breakpoints)[0] / len(reference) + eps
    cur_pct = np.histogram(current, bins=breakpoints)[0] / len(current) + eps
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

# Simulate drift
np.random.seed(42)
reference_dist = np.random.normal(0, 1, 1000)
no_drift = np.random.normal(0, 1, 1000)
mild_drift = np.random.normal(0.3, 1.2, 1000)
severe_drift = np.random.normal(1.0, 1.5, 1000)

print("Data Drift Detection (PSI):")
print("=" * 40)
for name, dist in [('No drift', no_drift), ('Mild drift', mild_drift), ('Severe drift', severe_drift)]:
    psi = calculate_psi(reference_dist, dist)
    status = 'OK' if psi < 0.1 else ('WARN' if psi < 0.25 else 'ALERT')
    print(f"  {name:15s}: PSI={psi:.4f} [{status}]")


Data Drift Detection (PSI):
  No drift       : PSI=0.0154 [OK]
  Mild drift     : PSI=0.1399 [WARN]
  Severe drift   : PSI=0.9742 [ALERT]


In [4]:
import matplotlib
matplotlib.use('Agg')
print('VERIFICATION PASSED: Phase 16.5 complete')


VERIFICATION PASSED: Phase 16.5 complete
